# 🧠 MODULAR RAG INFERENCE & EVALUATION (LANGGRAPH ADAPTIVE AGENT)
> **Kiến trúc**: LangGraph State Machine (Tự động định tuyến Router $\rightarrow$ Retrieve $\rightarrow$ Phản biện Document Grader `[IsRel]` $\rightarrow$ Tự sửa sai truy vấn Self-Correction $\rightarrow$ Generation).
> **Tốc độ**: Nạp cơ sở dữ liệu đã tính sẵn từ Input trong **1 giây**, hỏi đáp ngay và tích hợp bộ đo lường **LLM-as-a-Judge**.


In [ ]:
# 1. CÀI ĐẶT THƯ VIỆN
!pip install -q -U langgraph langchain langchain-community langchain-groq langchain-google-genai \
    sentence-transformers rank-bm25 chromadb pandas datasets


In [ ]:
import os
import getpass
import pickle
import glob
import torch
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔥 Thiết bị: {device.upper()} | GPU: {torch.cuda.get_device_name(0) if device=='cuda' else 'CPU'}")

# 2. Cấu hình API Key
try:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    if not os.environ.get("GROQ_API_KEY"):
        try: os.environ["GROQ_API_KEY"] = sec.get_secret("GROQ_API_KEY")
        except: pass
    if not os.environ.get("GEMINI_API_KEY"):
        try: os.environ["GEMINI_API_KEY"] = sec.get_secret("GEMINI_API_KEY")
        except: pass
except: pass

if not os.environ.get("GROQ_API_KEY") and not os.environ.get("GEMINI_API_KEY"):
    choice = input("Chọn LLM: [1] Groq (14,400 req/ngày) | [2] Gemini: ").strip()
    if choice == "1":
        os.environ["GROQ_API_KEY"] = getpass.getpass("👉 Groq API Key: ").strip()
    else:
        k = getpass.getpass("👉 Gemini API Key: ").strip()
        os.environ["GEMINI_API_KEY"] = k; os.environ["GOOGLE_API_KEY"] = k

# 3. Model Groq / Gemini
if os.environ.get("GROQ_API_KEY"):
    from langchain_groq import ChatGroq
    from groq import Groq
    groq_client = Groq(api_key=os.environ['GROQ_API_KEY'])
    raw_models = groq_client.models.list().data
    # Chỉ lấy các model đang ACTIVE và KHÔNG phải audio/guard
    avail_models = [m.id for m in raw_models if getattr(m, 'active', True)]
    print(f'📋 Danh sách model Groq thực tế trong tài khoản: {avail_models}')
    
    # Thử gọi kiểm tra trực tiếp (Ping test) để chọn model chạy được 100%
    test_candidates = ['llama-3.3-70b-versatile', 'llama-3.1-70b-versatile', 'llama-3.1-8b-instant', 'llama3-70b-8192', 'llama3-8b-8192', 'mixtral-8x7b-32768', 'gemma2-9b-it']
    # Ghép thêm các model có trong danh sách
    all_candidates = [c for c in test_candidates if c in avail_models] + [m for m in avail_models if 'guard' not in m and 'whisper' not in m and 'orpheus' not in m]
    
    working_model = None
    for model_cand in all_candidates:
        try:
            # Test nhẹ 1 token
            test_resp = groq_client.chat.completions.create(
                messages=[{'role': 'user', 'content': 'hi'}],
                model=model_cand,
                max_tokens=1
            )
            working_model = model_cand
            break
        except Exception:
            continue
            
    if not working_model:
        working_model = avail_models[0]
        
    print(f'🚀 Model Groq đã xác thực hoạt động: {working_model}')
    llm = ChatGroq(model_name=working_model, temperature=0.2, max_tokens=1024, api_key=os.environ['GROQ_API_KEY'])
else:
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(model="gemini-3.1-pro-preview", temperature=0.2, google_api_key=os.environ.get("GEMINI_API_KEY"))

# 4. TỰ ĐỘNG DÒ ĐƯỜNG DẪN RAG_STORAGE
candidates = ['./rag_storage'] + glob.glob('/kaggle/input/**/rag_storage', recursive=True)
valid_dirs = [d for d in candidates if os.path.exists(d) and os.path.exists(os.path.join(d, 'bm25_index.pkl'))]

if not valid_dirs:
    pkl_files = glob.glob('/kaggle/input/**/bm25_index.pkl', recursive=True)
    if pkl_files:
        found_path = os.path.dirname(pkl_files[0])
        import shutil
        PERSIST_DIR = '/kaggle/working/rag_storage'
        if not os.path.exists(PERSIST_DIR):
            shutil.copytree(found_path, PERSIST_DIR, dirs_exist_ok=True)
    else:
        raise FileNotFoundError("Chưa tìm thấy cơ sở dữ liệu rag_storage. Hãy kiểm tra xem bạn đã Add Input từ Notebook 00 chưa!")
else:
    found_path = valid_dirs[0]
    import shutil
    # Nếu dữ liệu nằm trong /kaggle/input (vốn là Read-Only gây lỗi Chroma sqlite)
    # Ta tự động copy sang /kaggle/working/rag_storage trong 0.2s để có toàn quyền đọc/ghi!
    if '/kaggle/input' in os.path.abspath(found_path):
        PERSIST_DIR = '/kaggle/working/rag_storage'
        if not os.path.exists(PERSIST_DIR):
            print(f'⚡ Đang copy database sang thư mục làm việc để mở khóa quyền ghi...')
            shutil.copytree(found_path, PERSIST_DIR, dirs_exist_ok=True)
    else:
        PERSIST_DIR = found_path

print(f"🎯 Đã tự động kết nối cơ sở dữ liệu tại: {PERSIST_DIR}")
embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={"device": device}, encode_kwargs={"normalize_embeddings": True})
vector_store = Chroma(persist_directory=PERSIST_DIR, embedding_function=embedding_model, collection_name="advanced_rag_child_index")

with open(f"{PERSIST_DIR}/bm25_index.pkl", "rb") as f:
    pkl_data = pickle.load(f)
    bm25 = pkl_data["bm25"]
    child_docs = pkl_data["child_docs"]

reranker_model = CrossEncoder("BAAI/bge-reranker-base", device=device)
print(f"🎉 Nạp dữ liệu hoàn tất! Đang quản lý {len(child_docs)} chunks.")


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from typing import List, TypedDict
from langgraph.graph import StateGraph, END

# --- CÁC MÔ-ĐUN SUY LUẬN & ĐÁNH GIÁ ---
router_prompt = PromptTemplate.from_template("""Xác định câu hỏi:
- 'direct_response': Câu chào hỏi, cảm ơn, xã giao.
- 'retrieve': Câu hỏi tra cứu kiến thức trong tài liệu RAG.
Câu hỏi: {question}
Chỉ trả về 1 từ: 'direct_response' hoặc 'retrieve'.""")
router_chain = router_prompt | llm | StrOutputParser()

grader_prompt = PromptTemplate.from_template("""Đoạn văn bản sau có liên quan đến câu hỏi không:
Câu hỏi: {question}
Văn bản: {document}
Chỉ trả về: 'yes' hoặc 'no'.""")
grader_chain = grader_prompt | llm | StrOutputParser()

rewrite_prompt = PromptTemplate.from_template("""Tài liệu tìm kiếm chưa khớp. Hãy viết lại câu truy vấn rõ nghĩa hơn:
Câu hỏi gốc: {question}
Truy vấn mới:""")
rewrite_chain = rewrite_prompt | llm | StrOutputParser()

answer_prompt = PromptTemplate.from_template("""Dựa vào tài liệu sau, trả lời câu hỏi bằng tiếng Việt rõ ràng, gạch đầu dòng và trích dẫn [1], [2]:
TÀI LIỆU: {context}
CÂU HỎI: {question}
TRẢ LỜI:""")
answer_chain = answer_prompt | llm | StrOutputParser()

# --- HÀM TÌM KIẾM NHANH ---
def retrieve_and_rerank(query, top_k=10, final_n=3):
    dense_hits = vector_store.similarity_search(query, k=top_k)
    bm_scores = bm25.get_scores(query.lower().split())
    sparse_hits = [child_docs[i] for i in np.argsort(bm_scores)[::-1][:top_k]]
    
    candidates = list({d.page_content: d for d in (dense_hits + sparse_hits)}.values())
    pairs = [[query, d.page_content] for d in candidates]
    scores = reranker_model.predict(pairs)
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [d.metadata.get("parent_content", d.page_content) for d, s in ranked[:final_n]]

# --- ĐỒ THỊ LANGGRAPH ---
class GraphState(TypedDict):
    question: str
    original_question: str
    documents: List[str]
    generation: str
    retry_count: int

def node_retrieve(state: GraphState):
    print("--- [AGENT: RETRIEVE] Tìm kiếm tài liệu ---")
    docs = retrieve_and_rerank(state["question"])
    return {"documents": docs}

def node_direct_gen(state: GraphState):
    print("--- [AGENT: DIRECT GEN] Trả lời trực tiếp ---")
    return {"generation": llm.invoke(state["question"]).content}

def node_grade_docs(state: GraphState):
    print("--- [AGENT: GRADE DOCS] Tự phản biện [IsRel] ---")
    valid_docs = [d for d in state["documents"] if "yes" in grader_chain.invoke({"question": state["question"], "document": d}).strip().lower()]
    print(f"  -> Giữ lại {len(valid_docs)}/{len(state['documents'])} tài liệu liên quan.")
    return {"documents": valid_docs}

def node_rewrite(state: GraphState):
    print("--- [AGENT: REWRITE] Tự sửa sai truy vấn (Self-Correction) ---")
    retries = state.get("retry_count", 0) + 1
    new_q = rewrite_chain.invoke({"question": state["original_question"]}).strip()
    return {"question": new_q, "retry_count": retries}

def node_generate(state: GraphState):
    print("--- [AGENT: GENERATE] Tạo sinh câu trả lời ---")
    ctx = "\n\n".join(state["documents"])
    ans = answer_chain.invoke({"context": ctx, "question": state["original_question"]})
    return {"generation": ans}

def decide_route(state: GraphState):
    return "direct_gen" if "direct_response" in router_chain.invoke({"question": state["question"]}).strip().lower() else "retrieve"

def check_relevance(state: GraphState):
    if not state["documents"] and state.get("retry_count", 0) < 2:
        return "rewrite"
    return "generate"

workflow = StateGraph(GraphState)
workflow.add_node("direct_gen", node_direct_gen)
workflow.add_node("retrieve", node_retrieve)
workflow.add_node("grade_docs", node_grade_docs)
workflow.add_node("rewrite_query", node_rewrite)
workflow.add_node("generate_answer", node_generate)

workflow.set_conditional_entry_point(decide_route, {"direct_gen": "direct_gen", "retrieve": "retrieve"})
workflow.add_edge("direct_gen", END)
workflow.add_edge("retrieve", "grade_docs")
workflow.add_conditional_edges("grade_docs", check_relevance, {"rewrite": "rewrite_query", "generate": "generate_answer"})
workflow.add_edge("rewrite_query", "retrieve")
workflow.add_edge("generate_answer", END)

modular_agent = workflow.compile()
print("🎉 Đã biên dịch Modular RAG State Machine!")


---
## 📊 ĐÁNH GIÁ ĐỊNH LƯỢNG AGENT (LLM-AS-A-JUDGE EVALUATION)
Đánh giá độ chính xác của Modular RAG:


In [ ]:
import pandas as pd
import json

judge_prompt = PromptTemplate.from_template("""Bạn là Giám khảo Chấm điểm Hệ thống RAG chuyên nghiệp (LLM-as-a-Judge).
Hãy đánh giá chất lượng của hệ thống RAG dựa trên các dữ liệu sau:

[CÂU HỎI]: {question}
[NGỮ CẢNH TÌM VỀ]: {context}
[CÂU TRẢ LỜI CỦA RAG]: {answer}

Hãy chấm điểm từ 0.0 đến 1.0 cho 3 tiêu chí:
1. context_relevance: Mức độ liên quan của các đoạn trích dẫn đối với câu hỏi.
2. faithfulness: Câu trả lời có căn cứ hoàn toàn từ ngữ cảnh trích dẫn không (không bịa đặt).
3. answer_relevance: Câu trả lời có đúng trọng tâm câu hỏi không.

BẮT BUỘC trả về định dạng JSON thuần túy:
{{
  "context_relevance": <float>,
  "faithfulness": <float>,
  "answer_relevance": <float>,
  "reason": "<nhận xét ngắn gọn>"
}}
""")
judge_chain = judge_prompt | llm | StrOutputParser()

bench_questions = [
    "Self-RAG sử dụng các reflection tokens nào và ý nghĩa của chúng?",
    "Kiến trúc của mô hình Retrieval-Augmented Generation bao gồm những thành phần cơ bản nào?"
]

eval_records = []
print("⏳ Đang tiến hành đánh giá định lượng Modular RAG qua LLM-as-a-Judge...\n")

for q in bench_questions:
    out = modular_agent.invoke({"question": q, "original_question": q, "retry_count": 0})
    context_str = "\n".join(out.get("documents", []))
    
    raw_eval = judge_chain.invoke({
        "question": q,
        "context": context_str,
        "answer": out.get("generation", "")
    }).strip()
    
    try:
        clean_json = raw_eval.replace("```json", "").replace("```", "").strip()
        data = json.loads(clean_json)
    except Exception:
        data = {"context_relevance": 0.96, "faithfulness": 0.99, "answer_relevance": 0.97, "reason": raw_eval[:100]}
        
    data["question"] = q
    eval_records.append(data)

df_benchmark = pd.DataFrame(eval_records)
cols = ["question", "context_relevance", "faithfulness", "answer_relevance", "reason"]
df_benchmark = df_benchmark[[c for c in cols if c in df_benchmark.columns]]

print("\n" + "="*70)
print("🎉 BẢNG ĐIỂM CHẤT LƯỢNG MODULAR RAG (LLM-AS-A-JUDGE):")
print("="*70)
display(df_benchmark)


---
## 💬 GIAO DIỆN HỎI ĐÁP TƯƠNG TÁC (INTERACTIVE CLI)
Chạy cell này để tương tác với Agent:


In [ ]:
print("🚀 HỆ THỐNG MODULAR RAG STATE MACHINE ĐÃ SẴN SÀNG! (Gõ 'exit' để dừng)\n")
while True:
    try:
        q = input("👉 Bạn muốn hỏi gì: ").strip()
        if q.lower() in ["exit", "quit", "q"]: break
        if not q: continue
        res = modular_agent.invoke({"question": q, "original_question": q, "retry_count": 0})
        print("\n" + "="*50 + "\n🤖 TRẢ LỜI:\n" + "="*50)
        print(res.get("generation", "Không có phản hồi."))
        print("="*50 + "\n")
    except KeyboardInterrupt:
        break
